In [ ]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchvision.models import resnet18, resnet34, resnet50
import torchvision.transforms as T


class CFG:
    data_path   = "train.npz" #kaggle input path for train.npz
    arch        = "resnet18"
    num_classes = 9

    epochs      = 100
    batch_size  = 128
    lr          = 0.05                 
    momentum    = 0.9
    weight_decay= 5e-4
    milestones  = [75, 90]
    gamma       = 0.1
    warmup_ep   = 5                    
    grad_clip   = 1.0                  

    eps         = 8.0 / 255.0
    step_size   = 2.0 / 255.0
    pgd_steps   = 10

    beta        = 6.0
    label_smooth= 0.0
    ema_decay   = 0.999

    val_size    = 5000
    robust_eval_n = 1024
    eval_pgd_steps = 20

    seed        = 0
    num_workers = 2
    out_path    = "model_robust.pt"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Data
class AugDataset(Dataset):
    def __init__(self, images, labels, train=True):
        self.images, self.labels, self.train = images, labels, train
        self.aug = T.Compose([
            T.RandomCrop(32, padding=4, padding_mode="reflect"),
            T.RandomHorizontalFlip(),
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        x = self.images[i]
        if self.train:
            x = self.aug(x)
        return x, self.labels[i]


def stratified_split(labels, val_size, num_classes, seed):
    rng = np.random.default_rng(seed)
    per_class = val_size // num_classes
    val_idx = []
    for c in range(num_classes):
        idx = np.where(labels.numpy() == c)[0]
        rng.shuffle(idx)
        val_idx.extend(idx[:per_class].tolist())
    val_idx = np.array(val_idx)
    mask = np.ones(len(labels), dtype=bool)
    mask[val_idx] = False
    return np.where(mask)[0], val_idx


def load_data(cfg):
    d = np.load(cfg.data_path)
    images = torch.from_numpy(d["images"]).float() / 255.0
    labels = torch.from_numpy(d["labels"]).long()

    # data sanity: catch a bad file before training
    assert images.min() >= 0.0 and images.max() <= 1.0, "images not in [0,1]"
    assert torch.isfinite(images).all(), "non-finite values in images"
    assert images.shape[1:] == (3, 32, 32), images.shape

    tr, va = stratified_split(labels, cfg.val_size, cfg.num_classes, cfg.seed)
    train_ds = AugDataset(images[tr], labels[tr], train=True)
    val_x, val_y = images[va], labels[va]
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
    print(f"train={len(tr)}  val={len(va)}  img={tuple(images.shape)}  "
          f"labels {labels.min().item()}..{labels.max().item()}  "
          f"pix[min/max]={images.min():.3f}/{images.max():.3f}")
    return train_loader, val_x, val_y


# Model + EMA
def build_model(cfg):
    ctor = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}[cfg.arch]
    m = ctor(weights=None)
    m.fc = nn.Linear(m.fc.in_features, cfg.num_classes)
    return m.to(DEVICE)


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, p in zip(self.shadow.state_dict().values(),
                        model.state_dict().values()):
            if s.dtype.is_floating_point:
                s.mul_(self.decay).add_(p, alpha=1 - self.decay)
            else:
                s.copy_(p)


# TRADES loss
def trades_loss(model, x, y, cfg):
    model.eval()
    criterion_kl = nn.KLDivLoss(reduction="sum")
    bs = x.size(0)

    x_adv = x.detach() + 0.001 * torch.randn_like(x)
    x_adv = torch.clamp(x_adv, 0.0, 1.0)

    with torch.no_grad():
        p_nat = F.softmax(model(x), dim=1).clamp_min(1e-8)

    for _ in range(cfg.pgd_steps):
        x_adv.requires_grad_(True)
        loss_kl = criterion_kl(F.log_softmax(model(x_adv), dim=1), p_nat)
        grad = torch.autograd.grad(loss_kl, x_adv)[0]
        x_adv = x_adv.detach() + cfg.step_size * grad.sign()
        x_adv = torch.min(torch.max(x_adv, x - cfg.eps), x + cfg.eps)
        x_adv = torch.clamp(x_adv, 0.0, 1.0)

    model.train()
    logits_nat = model(x)
    logits_adv = model(x_adv)

    loss_natural = F.cross_entropy(logits_nat, y, label_smoothing=cfg.label_smooth)
    loss_robust = (1.0 / bs) * criterion_kl(
        F.log_softmax(logits_adv, dim=1),
        F.softmax(logits_nat, dim=1).clamp_min(1e-8))
    return loss_natural + cfg.beta * loss_robust


# Eval
@torch.no_grad()
def clean_acc(model, x, y, bs=512):
    model.eval()
    correct = 0
    for i in range(0, len(x), bs):
        pred = model(x[i:i+bs].to(DEVICE)).argmax(1).cpu()
        correct += (pred == y[i:i+bs]).sum().item()
    return correct / len(x)


def pgd_acc(model, x, y, cfg, steps=None, bs=256):
    steps = steps or cfg.eval_pgd_steps
    model.eval()
    correct = 0
    for i in range(0, len(x), bs):
        xb = x[i:i+bs].to(DEVICE); yb = y[i:i+bs].to(DEVICE)
        x_adv = torch.clamp(xb + torch.empty_like(xb).uniform_(-cfg.eps, cfg.eps), 0, 1)
        for _ in range(steps):
            x_adv.requires_grad_(True)
            loss = F.cross_entropy(model(x_adv), yb)
            grad = torch.autograd.grad(loss, x_adv)[0]
            x_adv = x_adv.detach() + cfg.step_size * grad.sign()
            x_adv = torch.min(torch.max(x_adv, xb - cfg.eps), xb + cfg.eps)
            x_adv = torch.clamp(x_adv, 0, 1)
        with torch.no_grad():
            correct += (model(x_adv).argmax(1) == yb).sum().item()
    return correct / len(x)


def lr_at(cfg, epoch):
    if epoch < cfg.warmup_ep:
        return cfg.lr * (epoch + 1) / cfg.warmup_ep
    lr = cfg.lr
    for m in cfg.milestones:
        if epoch >= m:
            lr *= cfg.gamma
    return lr


# Train
def main(cfg=CFG):
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    train_loader, val_x, val_y = load_data(cfg)
    ridx = torch.randperm(len(val_x))[:cfg.robust_eval_n]
    rval_x, rval_y = val_x[ridx], val_y[ridx]

    torch.manual_seed(cfg.seed)
    model = build_model(cfg)
    ema = EMA(model, cfg.ema_decay)
    opt = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum,
                          weight_decay=cfg.weight_decay)

    best_score, best_state = -1.0, None
    for epoch in range(cfg.epochs):
        lr = lr_at(cfg, epoch)
        for g in opt.param_groups:
            g["lr"] = lr

        model.train()
        running, n_ok, n_bad = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = trades_loss(model, xb, yb, cfg)

            if not torch.isfinite(loss):       
                n_bad += 1
                if n_bad > 50:
                    raise RuntimeError("Too many non-finite losses; training unstable. "
                                       "Lower CFG.lr / CFG.beta, raise CFG.warmup_ep.")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            ema.update(model)
            running += loss.item(); n_ok += 1

        c_raw = clean_acc(model, val_x, val_y)
        r_raw = pgd_acc(model, rval_x, rval_y, cfg)
        c_ema = clean_acc(ema.shadow, val_x, val_y)
        r_ema = pgd_acc(ema.shadow, rval_x, rval_y, cfg)
        s_raw, s_ema = 0.5*c_raw + 0.5*r_raw, 0.5*c_ema + 0.5*r_ema

        if s_ema >= s_raw:
            score, tag, state = s_ema, "EMA", copy.deepcopy(ema.shadow.state_dict())
        else:
            score, tag, state = s_raw, "RAW", copy.deepcopy(model.state_dict())

        star = ""
        if score > best_score:
            best_score, best_state = score, state
            torch.save(best_state, cfg.out_path); star = " *saved*"

        print(f"ep {epoch:3d} lr {lr:.4f} loss {running/max(n_ok,1):.3f} "
              f"(bad {n_bad}) | raw c/r {c_raw:.3f}/{r_raw:.3f}  "
              f"ema c/r {c_ema:.3f}/{r_ema:.3f} | [{tag}] score {score:.4f} "
              f"(best {best_score:.4f}){star}")

    print(f"\nBest score {best_score:.4f}  ->  saved to {cfg.out_path}")
    sanity_check(cfg)


def sanity_check(cfg):
    ctor = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}[cfg.arch]
    m = ctor(weights=None)
    m.fc = nn.Linear(m.fc.in_features, cfg.num_classes)
    m.load_state_dict(torch.load(cfg.out_path, map_location="cpu"))
    m.eval()
    with torch.no_grad():
        out = m(torch.randn(1, 3, 32, 32))
    assert out.shape == (1, cfg.num_classes), out.shape
    print(f"Sanity fine: loads into stock {cfg.arch}, output {tuple(out.shape)}")


if __name__ == "__main__":
    main()